## San Diego VMT Study

# Import Libraries

In [ ]:
import numpy as np # Core numerical and array operations
import pandas as pd # Data table and analysis operations
import matplotlib.pyplot as plt # Plotting; syntax and appearance similar to Matlab
import pathlib # file handling robust to switching between Mac, PC, and Linux platforms
import geopandas as gpd # GIS functionality added to pandas. We will use lightly here to make a map.

# VMT Data Import

In [ ]:
cwd = pathlib.Path().cwd()
cwd


In [ ]:
inputdir = cwd / 'inputdata' # pathlib abstracts the Mac vs. PC syntax difference using '/' not as a string
inputdir

In [ ]:
datafile = inputdir / 'CARB-UCB-CA_VMT_Estimates_DataVintage_Oct2025_PRELIMINARY_FOR_STUDENT_USE.csv'


# View Data

In [ ]:
data = pd.read_csv(datafile)
data.head(10)


# Clean GEOID Column

In [ ]:
data['GEOID'] = data['GEOID'].map(str).str.zfill(12)
data.head()

In [ ]:
data_cleaned = data.set_index('GEOID')[['VMT_diff_state']]
data_cleaned.head()

In [ ]:
data_cleaned.describe()

# Calculate State-wide VMT Baseline using EMFAC (LDA, LDT1, LDT2, MCY)

In [ ]:
VMT_state = 716/39.5 # sum of VMT from downloaded spreadsheet divided by California population

In [ ]:
vmt = data_cleaned + VMT_state # important distinction between EMFAC (VMT_state), which is VMT driven within the boundaries of the state 
# and neighborhood data which are miles driven by residents of the state (data_clearned)

vmt = vmt.rename({'VMT_diff_state': 'VMTpc'}, axis=1)
vmt.describe()

# Visualization

In [ ]:
ca_cbg_shapefile_path = inputdir / 'cb_2019_06_bg_500k' / 'cb_2019_06_bg_500k.shp'
gdf = gpd.read_file(ca_cbg_shapefile_path) # Basic geopandas read; it will figure out it is a shapefile format.
gdf.head() # dataframe functions are inherited by geodataframes, in addition to geodataframe specific functions. 

In [ ]:
gdf.plot()

In [ ]:
gdf['ALAND'].sum() / 1e6 # convert m^2 to km^2

# Extract San Diego Data

In [ ]:
sd_shape = gdf[gdf['COUNTYFP'] == '073']
len(gdf)


In [ ]:
city_boundaries = gpd.read_file(
    inputdir / 'municipal_boundaries.shp'
)

In [ ]:
city_boundaries.columns

In [ ]:
sd_shape.plot()

# Merge

In [ ]:
merged = sd_shape.merge(vmt, left_on='GEOID', right_index=True, how='inner')
len(merged)

In [ ]:
merged.head()

In [ ]:
sd_vmt = merged.set_index('GEOID')[['VMTpc', 'geometry']]
sd_vmt.head()

# Map

In [ ]:
sd_vmt.plot(column='VMTpc', legend=True, cmap='jet', vmin=4, vmax=15)


# Merge with ACS population and housing Data

### Census API key required

As of 2026, the U.S. Census Bureau requires an API key for Census Data API queries. Request a free key at https://api.census.gov/data/key_signup.html. When you run the next cell, paste the key into the hidden prompt. The key is **not saved into this notebook**.


In [ ]:
import json
import os
import getpass
import urllib.parse
import urllib.request
import urllib.error

def get_census_api_key():
    """Use an environment variable if available; otherwise prompt securely."""
    key = os.environ.get("CENSUS_API_KEY")
    if key:
        return key.strip()

    key = getpass.getpass(
        "Paste your Census API key (input will be hidden): "
    ).strip()

    if not key:
        raise ValueError(
            "A Census API key is required. Request a free key from "
            "https://api.census.gov/data/key_signup.html"
        )

    return key


def load_ca_acs_block_groups_2019(api_key):
    """Pull 2019 ACS 5-year population and housing for California block groups."""
    base_url = "https://api.census.gov/data/2019/acs/acs5"

    params = [
        ("get", "NAME,B01003_001E,B25001_001E"),
        ("for", "block group:*"),
        ("in", "state:06"),
        ("in", "county:*"),
        ("in", "tract:*"),
        ("key", api_key),
    ]

    url = base_url + "?" + urllib.parse.urlencode(params)

    try:
        with urllib.request.urlopen(url) as response:
            body = response.read().decode("utf-8")
    except urllib.error.HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Census API request failed with HTTP {exc.code}: {error_body}"
        ) from exc

    if not body.lstrip().startswith("["):
        raise RuntimeError(
            "Census API did not return JSON. Response was:\n"
            + body[:1000]
        )

    raw = json.loads(body)
    acs = pd.DataFrame(raw[1:], columns=raw[0])

    acs["GEOID"] = (
        acs["state"].astype(str)
        + acs["county"].astype(str)
        + acs["tract"].astype(str)
        + acs["block group"].astype(str)
    )

    acs["Population"] = pd.to_numeric(acs["B01003_001E"], errors="coerce")
    acs["Housing Units"] = pd.to_numeric(acs["B25001_001E"], errors="coerce")

    return acs.set_index("GEOID")


# Census now requires an API key for data queries.
census_api_key = get_census_api_key()

acs_bg = load_ca_acs_block_groups_2019(census_api_key)

pop = acs_bg[["Population"]].copy()
housing_units = acs_bg[["Housing Units"]].copy()

print("ACS block-group rows:", len(acs_bg))
print(
    "San Diego ACS block-group rows:",
    acs_bg.index.str.startswith("06073").sum()
)
print("Example ACS GEOIDs:", list(acs_bg.index[:5]))


In [ ]:
# Population and housing units were loaded above directly from the 2019 Census API.


Trim Geoid 15000000US Prefix

In [ ]:
print("Population GEOIDs:", list(pop.index[:5]))
print("Housing GEOIDs:", list(housing_units.index[:5]))
print("VMT GEOIDs:", list(vmt.index[:5]))

print("\nSan Diego population rows:", pop.index.str.startswith('06073').sum())
print("San Diego housing rows:", housing_units.index.str.startswith('06073').sum())
print("San Diego VMT rows:", vmt.index.str.startswith('06073').sum())


Set GEOID as Index, rename column of interest, and drop remainder

In [ ]:
pop.head()


Function to combine Housing + Population Data 

In [ ]:
# Census helper functions are defined above so local files are selected before loading.


In [ ]:
pop['Population'].sum()


In [ ]:
housing_units['Housing Units'].sum()


In [ ]:
housing_units.head()


Merge VMT, Population, Housing Units

In [ ]:
# Merge the VMT data with Census population and housing by 12-digit GEOID.
vmt_merged = (
    vmt
    .merge(pop, left_index=True, right_index=True, how="inner")
    .merge(housing_units, left_index=True, right_index=True, how="inner")
)

print("VMT rows:", len(vmt))
print("Merged statewide rows:", len(vmt_merged))
print(
    "Merged San Diego County rows:",
    vmt_merged.index.str.startswith("06073").sum()
)
print(
    "Merged San Diego County population:",
    vmt_merged.loc[
        vmt_merged.index.str.startswith("06073"),
        "Population"
    ].sum()
)

vmt_merged.head()


In [ ]:
len(vmt_merged)


In [ ]:
# Population and housing tables were loaded above from the validated local block-group files.


In [ ]:
print("VMT rows:", len(vmt))
print("Population rows:", len(pop))
print("Housing rows:", len(housing_units))
print("Merged rows:", len(vmt_merged))

print("\nMerged San Diego rows:", vmt_merged.index.str.startswith('06073').sum())
print("Merged San Diego population:",
      vmt_merged.loc[vmt_merged.index.str.startswith('06073'), 'Population'].sum())


In [ ]:
print(pop.index[:5])
print(vmt.index[:5])

# Statistics

In [ ]:
# vmt_merged was already built above from the validated local Census files.
vmt_merged = vmt_merged.copy()


In [ ]:
print("VMT rows:", len(vmt))
print("Population rows:", len(pop))
print("Housing rows:", len(housing_units))
print("Merged rows:", len(vmt_merged))

print("\nPopulation sum:", vmt_merged['Population'].sum())
print("Missing population:", vmt_merged['Population'].isna().sum())

print("\nFirst VMT GEOIDs:")
print(vmt.index[:5])

print("\nFirst Population GEOIDs:")
print(pop.index[:5])

print("\nFirst Housing GEOIDs:")
print(housing_units.index[:5])

In [ ]:
print("VMT rows:", len(vmt))
print("Population rows:", len(pop))
print("Housing rows:", len(housing_units))
print("Merged rows:", len(vmt_merged))

In [ ]:
# Step 1: VMT + population only
test_pop_merge = vmt.merge(
    pop,
    left_index=True,
    right_index=True,
    how='inner'
)

print("VMT rows:", len(vmt))
print("Population rows:", len(pop))
print("VMT + Population matched rows:", len(test_pop_merge))

# Step 2: add housing units
test_full_merge = test_pop_merge.merge(
    housing_units,
    left_index=True,
    right_index=True,
    how='inner'
)

print("Housing rows:", len(housing_units))
print("After adding Housing Units:", len(test_full_merge))

print("\nVMT IDs:")
print(vmt.index[:5])

print("\nPopulation IDs:")
print(pop.index[:5])

print("\nHousing IDs:")
print(housing_units.index[:5])

In [ ]:
# Population-weighted statewide VMT.
statewide_valid = vmt_merged[["VMTpc", "Population"]].dropna().copy()
statewide_valid = statewide_valid[statewide_valid["Population"] > 0]

if len(statewide_valid) == 0:
    raise ValueError(
        "The Census API returned data, but no rows matched the VMT GEOIDs."
    )

statewide_mean_vmt_calc = np.average(
    statewide_valid["VMTpc"],
    weights=statewide_valid["Population"]
)

print("Valid statewide rows:", len(statewide_valid))
print("Statewide population represented:", statewide_valid["Population"].sum())
print("Population-weighted statewide VMT:", statewide_mean_vmt_calc)

statewide_mean_vmt_calc


In [ ]:
statewide_difference_from_emfac = statewide_mean_vmt_calc - VMT_state

print("Population-weighted statewide VMT:", statewide_mean_vmt_calc)
print("EMFAC statewide VMT baseline:", VMT_state)
print("Difference:", statewide_difference_from_emfac)

statewide_difference_from_emfac


Histogram

In [ ]:
# Plot the statewide data
plt.hist(vmt_merged['VMTpc'], weights=vmt_merged['Population'] / 1e6, bins=np.arange(3, 30), alpha=0.5, label='California')
# Set bins to be 1 VMT from 3 to 30, use millions of people for plotting convenience.

# Filter San Diego County (state FIPS 06, county FIPS 073)
sd_vmt_merged = vmt_merged[vmt_merged.index.str.startswith('06073')]
plt.hist(sd_vmt_merged['VMTpc'], weights=sd_vmt_merged['Population'] / 1e6, bins=np.arange(3, 30), color='orange', alpha=0.8, label='San Diego')

plt.xlabel('VMT/person/day')
plt.ylabel('Million People')
plt.title('VMT Distribution')
plt.legend()



In [ ]:
vmt_merged = vmt_merged.merge(gdf[['GEOID', 'ALAND']], left_index=True, right_on='GEOID', how='inner')
vmt_merged = vmt_merged.set_index('GEOID')
vmt_merged.head()

In [ ]:
vmt_merged['Population Density'] = vmt_merged['Population'] / vmt_merged['ALAND'] * 1e6 * 1.61**2
# people/m^2 * m^2/km^2 * km^2/mi^2 = people/mi^2.
vmt_merged.describe()

In [ ]:
# Scatterplot
# First filter to rows with > 0 pop density. We'll use > 1 to avoid small floating point issues.
vmt_subset = vmt_merged[vmt_merged['Population Density'] > 1]

# Plot the statewide data
plt.scatter(np.log10(vmt_subset['Population Density']), vmt_subset['VMTpc'], s=1, alpha=0.5, label='California')

# San Diego data overlaid
sd_vmt_subset = vmt_subset[vmt_subset.index.str.startswith('06073')]
plt.scatter(np.log10(sd_vmt_subset['Population Density']), sd_vmt_subset['VMTpc'], s=1, color='orange', alpha=0.8, label='San Diego')

plt.ylabel('VMT per person, per day')
plt.xlabel('Log$_{10}$ Population Density (people per sq mi)')
plt.legend()


In [ ]:
# Filter for San Diego County (state FIPS 06, county FIPS 073)
sd_vmt_merged = vmt_merged[vmt_merged.index.str.startswith('06073')]

plt.scatter(np.log10(sd_vmt_merged['Population Density'] + 1), sd_vmt_merged['VMTpc'], s=sd_vmt_merged['Population']/1e4, alpha=0.1)
plt.xlabel('Log{10}$ Population Density (people per sq mi)')
plt.ylabel('VMT per capita (miles/day)')
plt.title('San Diego County VMT vs. Population Density')
plt.show()


In [ ]:
np.corrcoef(np.log10(vmt_subset['Population Density']), vmt_subset['VMTpc'])

In [ ]:
len(sd_vmt)

In [ ]:
# Use the already-merged statewide dataset and filter to San Diego County.
# This avoids re-merging the map-only sd_vmt object with Census tables.

sd_vmt_stats = vmt_merged[
    vmt_merged.index.str.startswith('06073')
].copy()

print("San Diego rows available for statistics:", len(sd_vmt_stats))
print("San Diego population represented:", sd_vmt_stats['Population'].sum())


In [ ]:
sd_vmt.head()

In [ ]:
# San Diego County subset from the validated statewide merge.
sd_vmt_stats = vmt_merged[
    vmt_merged.index.str.startswith("06073")
].copy()

sd_valid = sd_vmt_stats[["VMTpc", "Population"]].dropna().copy()
sd_valid = sd_valid[sd_valid["Population"] > 0]

if len(sd_valid) == 0:
    raise ValueError(
        "No San Diego County block groups matched after the Census API merge."
    )

sd_popmean_vmt = np.average(
    sd_valid["VMTpc"],
    weights=sd_valid["Population"]
)

print("Valid San Diego County rows:", len(sd_valid))
print("San Diego County population represented:", sd_valid["Population"].sum())
print("Population-weighted San Diego County VMT:", sd_popmean_vmt)

sd_popmean_vmt


In [ ]:
sd_popmean_vmt - statewide_mean_vmt_calc


In [ ]:
deciles = np.arange(0, 1.1, 0.1)

sd_vmt_deciles = np.quantile(
    sd_valid['VMTpc'],
    deciles,
    weights=sd_valid['Population'],
    method='inverted_cdf'
)

sd_vmt_deciles = {
    round(deciles[x], 1): round(sd_vmt_deciles[x], 1)
    for x in range(len(deciles))
}

sd_vmt_deciles


Below here is where Chris will try to add in San Diego (City) Data

# City of San Diego VMT Analysis

In [ ]:
import zipfile
import urllib.request

# Official 2019 Census TIGER/Line California Places file
city_place_url = (
    "https://www2.census.gov/geo/tiger/TIGER2019/PLACE/"
    "tl_2019_06_place.zip"
)

city_place_zip = inputdir / "tl_2019_06_place.zip"
city_place_folder = inputdir / "tl_2019_06_place"
city_place_shp = city_place_folder / "tl_2019_06_place.shp"

# Download only if it is not already stored locally
if not city_place_zip.exists():
    urllib.request.urlretrieve(
        city_place_url,
        city_place_zip
    )

# Extract only if the shapefile has not already been extracted
if not city_place_shp.exists():
    city_place_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(city_place_zip, "r") as zip_ref:
        zip_ref.extractall(city_place_folder)

# Read California places
ca_places_2019 = gpd.read_file(city_place_shp)

print("California places loaded:", len(ca_places_2019))

In [ ]:
# San Diego city = PLACEFP 66000
sd_city_boundary = ca_places_2019[
    ca_places_2019["PLACEFP"] == "66000"
].copy()

print("City boundary rows:", len(sd_city_boundary))
print(sd_city_boundary[["NAME", "PLACEFP", "GEOID"]])

In [ ]:
# San Diego city = PLACEFP 66000
sd_city_boundary = ca_places_2019[
    ca_places_2019["PLACEFP"] == "66000"
].copy()

print("City boundary rows:", len(sd_city_boundary))
print(sd_city_boundary[["NAME", "PLACEFP", "GEOID"]])

In [ ]:
ax = sd_city_boundary.plot(
    figsize=(8, 8),
    facecolor="none",
    edgecolor="black"
)

ax.set_title("2019 City of San Diego Boundary")
ax.set_axis_off()

plt.show()

In [ ]:
# Start with a COPY so sd_shape itself is not modified
sd_bg_citycheck = sd_shape[
    ["GEOID", "geometry"]
].copy()

# Project both datasets to California Albers
sd_bg_citycheck = sd_bg_citycheck.to_crs(epsg=3310)
sd_city_boundary_proj = sd_city_boundary.to_crs(epsg=3310)

# Create one guaranteed-inside point for each block group
sd_bg_citycheck["rep_point"] = (
    sd_bg_citycheck.geometry.representative_point()
)

# There is only one San Diego City polygon record
sd_city_geometry = (
    sd_city_boundary_proj.geometry.iloc[0]
)

# Determine whether each block group's representative point
# falls inside San Diego City
sd_bg_citycheck["in_sd_city"] = (
    sd_bg_citycheck["rep_point"]
    .within(sd_city_geometry)
)

# Save the GEOIDs assigned to the City
sd_city_geoids = sd_bg_citycheck.loc[
    sd_bg_citycheck["in_sd_city"],
    "GEOID"
].astype(str)

print(
    "San Diego County block groups:",
    len(sd_bg_citycheck)
)

print(
    "Block groups assigned to City of San Diego:",
    len(sd_city_geoids)
)

In [ ]:
### City Geography Method

In [ ]:
sd_city_vmt = vmt_merged[
    vmt_merged.index.isin(sd_city_geoids)
].copy()

print("City VMT rows:", len(sd_city_vmt))
print(
    "Population represented:",
    sd_city_vmt["Population"].sum()
)

sd_city_vmt.head()

In [ ]:
sd_city_vmt.columns

In [ ]:
# Entire San Diego County
sd_county_compare = vmt_merged[
    vmt_merged.index.str.startswith("06073")
].copy()

# City of San Diego
sd_city_compare = sd_city_vmt.copy()

# San Diego County excluding City of San Diego
sd_rest_county_compare = sd_county_compare[
    ~sd_county_compare.index.isin(sd_city_geoids)
].copy()

print("County rows:", len(sd_county_compare))
print("City rows:", len(sd_city_compare))
print(
    "Rest of County rows:",
    len(sd_rest_county_compare)
)

In [ ]:
def population_weighted_vmt(df):
    valid = df[
        ["VMTpc", "Population"]
    ].dropna().copy()

    valid = valid[
        valid["Population"] > 0
    ]

    return np.average(
        valid["VMTpc"],
        weights=valid["Population"]
    )

In [ ]:
city_mean_vmt = population_weighted_vmt(
    sd_city_compare
)

county_mean_vmt = population_weighted_vmt(
    sd_county_compare
)

rest_county_mean_vmt = population_weighted_vmt(
    sd_rest_county_compare
)

print(
    "City of San Diego:",
    round(city_mean_vmt, 2)
)

print(
    "San Diego County overall:",
    round(county_mean_vmt, 2)
)

print(
    "Rest of San Diego County:",
    round(rest_county_mean_vmt, 2)
)

In [ ]:
city_county_summary = pd.DataFrame({
    "Geography": [
        "City of San Diego",
        "Rest of San Diego County",
        "San Diego County Overall"
    ],
    "Population": [
        sd_city_compare["Population"].sum(),
        sd_rest_county_compare["Population"].sum(),
        sd_county_compare["Population"].sum()
    ],
    "Housing Units": [
        sd_city_compare["Housing Units"].sum(),
        sd_rest_county_compare["Housing Units"].sum(),
        sd_county_compare["Housing Units"].sum()
    ],
    "VMT per Person per Day": [
        city_mean_vmt,
        rest_county_mean_vmt,
        county_mean_vmt
    ]
})

city_county_summary

In [ ]:
plt.figure(figsize=(8, 5))

bars = plt.bar(
    city_county_summary["Geography"],
    city_county_summary["VMT per Person per Day"]
)

plt.ylabel("VMT per Person per Day")
plt.title(
    "Resident VMT: City of San Diego vs. San Diego County"
)

plt.xticks(rotation=10)

for bar in bars:
    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.1f}",
        ha="center",
        va="bottom"
    )

plt.show()

In [ ]:
sd_county_vmt_map = sd_shape[
    ["GEOID", "geometry"]
].merge(
    sd_county_compare[["VMTpc"]],
    left_on="GEOID",
    right_index=True,
    how="inner"
)

sd_city_vmt_map = sd_shape[
    sd_shape["GEOID"].isin(sd_city_geoids)
][["GEOID", "geometry"]].merge(
    sd_city_compare[["VMTpc"]],
    left_on="GEOID",
    right_index=True,
    how="inner"
)

In [ ]:
vmt_map_min = (
    sd_county_compare["VMTpc"]
    .quantile(0.05)
)

vmt_map_max = (
    sd_county_compare["VMTpc"]
    .quantile(0.95)
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 8)
)

sd_county_vmt_map.plot(
    column="VMTpc",
    cmap="viridis",
    vmin=vmt_map_min,
    vmax=vmt_map_max,
    legend=True,
    ax=axes[0]
)

axes[0].set_title(
    "San Diego County VMT per Person"
)
axes[0].set_axis_off()


sd_city_vmt_map.plot(
    column="VMTpc",
    cmap="viridis",
    vmin=vmt_map_min,
    vmax=vmt_map_max,
    legend=True,
    ax=axes[1]
)

axes[1].set_title(
    "City of San Diego VMT per Person"
)
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
bins = np.arange(
    np.floor(
        sd_county_compare["VMTpc"].min()
    ),
    np.ceil(
        sd_county_compare["VMTpc"].max()
    ) + 1,
    1
)

city_weights = (
    sd_city_compare["Population"]
    / sd_city_compare["Population"].sum()
    * 100
)

rest_county_weights = (
    sd_rest_county_compare["Population"]
    / sd_rest_county_compare["Population"].sum()
    * 100
)

plt.hist(
    sd_city_compare["VMTpc"],
    bins=bins,
    weights=city_weights,
    alpha=0.6,
    label="City of San Diego"
)

plt.hist(
    sd_rest_county_compare["VMTpc"],
    bins=bins,
    weights=rest_county_weights,
    alpha=0.5,
    label="Rest of San Diego County"
)

plt.xlabel("VMT per Person per Day")
plt.ylabel("Percent of Population")
plt.title(
    "Resident VMT Distribution: City vs. Rest of County"
)
plt.legend()

plt.show()

In [ ]:
city_density = sd_city_compare[
    sd_city_compare["Population Density"] > 1
].copy()

rest_density = sd_rest_county_compare[
    sd_rest_county_compare["Population Density"] > 1
].copy()

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    np.log10(
        city_density["Population Density"]
    ),
    city_density["VMTpc"],
    s=8,
    alpha=0.4,
    label="City of San Diego"
)

plt.scatter(
    np.log10(
        rest_density["Population Density"]
    ),
    rest_density["VMTpc"],
    s=8,
    alpha=0.3,
    label="Rest of San Diego County"
)

plt.xlabel(
    "Log$_{10}$ Population Density (people per sq mi)"
)
plt.ylabel("VMT per Person per Day")

plt.title(
    "Population Density and Resident VMT"
)

plt.legend()

plt.show()

In [ ]:
city_density_corr = np.corrcoef(
    np.log10(
        city_density["Population Density"]
    ),
    city_density["VMTpc"]
)[0, 1]

rest_density_corr = np.corrcoef(
    np.log10(
        rest_density["Population Density"]
    ),
    rest_density["VMTpc"]
)[0, 1]

print(
    "City correlation:",
    round(city_density_corr, 3)
)

print(
    "Rest of County correlation:",
    round(rest_density_corr, 3)
)

In [ ]:
city_vmt_deciles_array = np.quantile(
    sd_city_compare["VMTpc"],
    deciles,
    weights=sd_city_compare["Population"],
    method="inverted_cdf"
)

city_vmt_deciles = {
    round(deciles[x], 1):
    round(city_vmt_deciles_array[x], 1)
    for x in range(len(deciles))
}

city_vmt_deciles

In [ ]:
city_county_deciles = pd.DataFrame({
    "Percentile": list(
        sd_vmt_deciles.keys()
    ),
    "City of San Diego": list(
        city_vmt_deciles.values()
    ),
    "San Diego County": list(
        sd_vmt_deciles.values()
    )
})

city_county_deciles

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    city_county_deciles["Percentile"],
    city_county_deciles["City of San Diego"],
    marker="o",
    label="City of San Diego"
)

plt.plot(
    city_county_deciles["Percentile"],
    city_county_deciles["San Diego County"],
    marker="o",
    label="San Diego County"
)

plt.xlabel("Population Percentile")
plt.ylabel("VMT per Person per Day")
plt.title(
    "Population-Weighted VMT Percentiles"
)

plt.legend()
plt.show()

# VMT and CalEnviroScreen 5.0

This section examines whether resident vehicle miles traveled (VMT) differs
across communities with different levels of cumulative environmental burden,
using CalEnviroScreen 5.0.

Because the VMT dataset uses 2019 Census block groups while CalEnviroScreen
5.0 uses updated Census tract geography, VMT block groups are spatially
assigned to CalEnviroScreen tracts.

In [ ]:
from pathlib import Path

repo = Path(r"C:\Users\chris\OneDrive\Documents\GitHub\The858")
inputdir = repo / "inputdata"

print("Repository:", repo)
print("Input directory:", inputdir)
print("Input directory exists:", inputdir.exists())

all_shapefiles = list(inputdir.rglob("*.shp"))

print("\nShapefiles found:")
for shp in all_shapefiles:
    print(shp)

In [ ]:
from pathlib import Path
import geopandas as gpd

ces_shp = Path(
    r"C:\Users\chris\OneDrive\Documents\GitHub\The858\inputdata\CalEnviroScreen5\CES5_final_shapefile.shp"
)

ces5 = gpd.read_file(ces_shp)

print("CES 5.0 tracts:", len(ces5))
print("\nColumns:")
print(ces5.columns.tolist())

In [ ]:
print(ces5.columns.tolist())

In [ ]:
import pandas as pd

In [ ]:
# Make a clean working copy
ces5_clean = ces5.copy()

# Convert tract ID to an 11-digit Census tract GEOID
ces5_clean["tract_geoid"] = (
    pd.to_numeric(
        ces5_clean["tract"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
    .str.zfill(11)
)

# Make CES fields numeric
for col in [
    "CIscore",
    "CIscoreP",
    "PollutionP",
    "PopCharP"
]:
    ces5_clean[col] = pd.to_numeric(
        ces5_clean[col],
        errors="coerce"
    )

# San Diego County tract GEOIDs begin with 06073
ces5_sd = ces5_clean[
    ces5_clean["tract_geoid"].str.startswith("06073")
].copy()

print("California CES tracts:", len(ces5_clean))
print("San Diego County CES tracts:", len(ces5_sd))

ces5_sd[
    [
        "tract_geoid",
        "county",
        "CIscore",
        "CIscoreP",
        "PollutionP",
        "PopCharP"
    ]
].head(10)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

In [ ]:
ces5_clean = ces5.copy()

ces5_clean["tract_geoid"] = (
    pd.to_numeric(
        ces5_clean["tract"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
    .str.zfill(11)
)

for col in [
    "CIscore",
    "CIscoreP",
    "PollutionP",
    "PopCharP"
]:
    ces5_clean[col] = pd.to_numeric(
        ces5_clean[col],
        errors="coerce"
    )

ces5_sd = ces5_clean[
    ces5_clean["tract_geoid"].str.startswith("06073")
].copy()

print("California CES tracts:", len(ces5_clean))
print("San Diego County CES tracts:", len(ces5_sd))

ces5_sd[
    [
        "tract_geoid",
        "county",
        "CIscore",
        "CIscoreP",
        "PollutionP",
        "PopCharP"
    ]
].head(10)

In [ ]:
sd_vmt_ces = sd_shape[
    ["GEOID", "geometry"]
].merge(
    sd_county_compare[
        ["VMTpc", "Population"]
    ],
    left_on="GEOID",
    right_index=True,
    how="inner"
)

sd_vmt_ces["in_sd_city"] = (
    sd_vmt_ces["GEOID"].isin(sd_city_geoids)
)

print("San Diego VMT block groups:", len(sd_vmt_ces))
print(
    "City of San Diego VMT block groups:",
    sd_vmt_ces["in_sd_city"].sum()
)

# DP03 Economic Characteristics of San Diego County

From US Census Bureau DP03 (Economic Characteristics)
- 0004E (Total Employed)
- 0009PE Unemployment Rate
- 0062E Median Household Income
- 0119PE Percent of families below the poverty level
- 0019E Drove alone
- 0020E Carpooled
- 0021E Public Transportation (excludes rideshare)
- 0024E Worked from home
- 0025E Mean Travel time to work (in minutes)

In [ ]:
# target columns / when you download a ZIP file from the US census buereau, there is a file in there called -'column-metadata.csv' and that's how you can map the column "codes" to new column names so it's easiet to filter, etc
columns_to_keep_dp03 = { # when you use {} it's called a 'dictionary' and you map key:value pairs to each other. If you download a new dataset, I recommend setting up a dictionary like this because it's very configurable and easy to add/remove new columns
    'GEO_ID': 'GEOID', 
    'DP03_0062E': 'Median_Income',
    'DP03_0128PE': 'Percent_Poverty_All_People',
    'DP03_0009PE': 'Unemployment_Rate',
    'DP03_0019E': 'Commute_Drove_Alone',
    'DP03_0020E': 'Commute_Carpooled',
    'DP03_0021E': 'Commute_Transit',
    'DP03_0024E': 'Commute_WFH',
    'DP03_0025E': 'Mean_Commute_Time_Mins'
}

# read file using the pandas .read_csv that points to where the data is stored (the inputdir is defined further up in the notebook)
df_dp03 = pd.read_csv(inputdir / 'DP03/ACSDP5Y2019.DP03-Data.csv', skiprows=[1], usecols=columns_to_keep_dp03.keys())

# rename col - this essentially says for the data frame (df) df_dp03 rename the columns using the dictionary we mapped above
df_dp03 = df_dp03.rename(columns=columns_to_keep_dp03)

# strip US from GEOID as done with VMT
df_dp03['GEOID'] = df_dp03['GEOID'].str.split('US').str[1]
df_dp03 = df_dp03.set_index('GEOID')

# if you change the number in the paranthesis it will show that number of rows in the dataframe. It defaults to 5 if you don't add a number.
df_dp03.head()


Plots for DP03

In [ ]:
sd_shape['Tract_GEOID'] = sd_shape['GEOID'].str[:11] # strips first 11 characters using some neat splicing

sd_demographics = sd_shape.merge(df_dp03, left_on='Tract_GEOID', right_index=True, how='inner') # this is merging two datasets, our dp03 dataframe with the demographics dataframe that has the GEOID data so we can plot it

sd_demographics['Median_Income'] = pd.to_numeric(sd_demographics['Median_Income'], errors='coerce') # this is to 'coerce' a text value to a numeric value so that you can perform calculations on it. Essentially just formatting a string to an integer.
sd_demographics.plot(
    column='Median_Income', # you can swap this variable out with any one of the "Columns to Keep" to update the chart below, or just copy this whole code block and create a brand new chart. Don't forget to also update where it says 'Median Income' in the coerce section.
    legend=True, 
    cmap='viridis', # stylistic choice
    figsize=(12, 8), # how big do you want the chart     
    edgecolor='black', # style choice for boundary
    linewidth=0.1, # thickness of the boundary line  
    missing_kwds={'color': 'lightgrey'}, # this handles any census blocks that don't have data, they will show up light grey (kwds is python for keywords)
    legend_kwds={'label': 'Median Income ($)'}
)



# DP04 Housing Characteristics of San Diego County

In [ ]:
# this follows the exact same process as above but for the DP04 data set from the Census Bureau, these are some of the interesting characteristics we can use for the VMT analysis and where housing units are most prevalent
# target columns for DP04 (Housing Characteristics)
columns_to_keep_dp04 = { 
    'GEO_ID': 'GEOID', 
    'DP04_0001E': 'Total_Housing_Units',
    'DP04_0017E': 'Built_2014_or_Later',
    'DP04_0018E': 'Built_2010_to_2013',
    'DP04_0019E': 'Built_2000_to_2009',
    'DP04_0007E': 'Single_Family_Detached',
    'DP04_0008E': 'Single_Family_Attached',
    'DP04_0013E': 'Multi_Family_20_Plus',
    'DP04_0058E': 'Zero_Car_Households',
    'DP04_0059E': 'One_Car_Households',
    'DP04_0060E': 'Two_Car_Households',
    'DP04_0061E': 'Three_Plus_Car_Households',
    'DP04_0046E': 'Owner_Occupied',
    'DP04_0047E': 'Renter_Occupied'
}

df_dp04 = pd.read_csv(inputdir / 'DP04/ACSDP5Y2019.DP04-Data.csv', skiprows=[1], usecols=columns_to_keep_dp04.keys())

df_dp04 = df_dp04.rename(columns=columns_to_keep_dp04)

df_dp04['GEOID'] = df_dp04['GEOID'].str.split('US').str[1]
df_dp04 = df_dp04.set_index('GEOID')

df_dp04.head()


In [ ]:
sd_housing = sd_shape.merge(df_dp04, left_on='Tract_GEOID', right_index=True, how='inner')

sd_housing['Total_Housing_Units'] = pd.to_numeric(sd_housing['Total_Housing_Units'], errors='coerce')

sd_housing.plot(
    column='Total_Housing_Units', 
    legend=True, 
    cmap='Purples', 
    figsize=(12, 8),      
    edgecolor='black', 
    linewidth=0.1,   
    missing_kwds={'color': 'lightgrey'}, 
    legend_kwds={'label': 'Total Housing Units'}
)